In [8]:
from faker import Faker
import pandas as pd
import random
from datetime import timedelta


# Configuration


fake = Faker("en_IN")
Faker.seed(42)
random.seed(42)

NUM_CUSTOMERS = 500

loyalty_levels = ["Silver", "Gold", "Platinum"]

# generating customers

customers = []

for customer_id in range(1001, 1001 + NUM_CUSTOMERS):

    join_date = fake.date_between(
        start_date="-3y",
        end_date="-90d"
    )

    created_at = fake.date_time_between(
        start_date=join_date,
        end_date="now"
    )

    last_updated = created_at + timedelta(
        days=random.randint(0, 120),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )

    customer = {
        "Customer_ID": customer_id,
        "Customer_Name": fake.name(),
        "Email": fake.email(),
        "Phone": fake.msisdn()[-10:],
        "Gender": random.choice(["Male", "Female"]),
        "Date_of_Birth": fake.date_of_birth(
            minimum_age=18,
            maximum_age=70
        ),
        "City": fake.city(),
        "State": fake.state(),
        "Country": "India",
        "Join_Date": join_date,
        "Loyalty_Status": random.choice(loyalty_levels),
        "Created_At": created_at,
        "Last_Updated": last_updated
    }

    customers.append(customer)

df = pd.DataFrame(customers)

#missing emails

missing_email = df.sample(frac=0.05, random_state=1).index
df.loc[missing_email, "Email"] = None
#missing phone numbers

missing_phone = df.sample(frac=0.02, random_state=2).index
df.loc[missing_phone, "Phone"] = None

#

for idx in df.sample(frac=0.06, random_state=3).index:

    state = df.loc[idx, "State"]

    style = random.choice([
        "upper",
        "lower",
        "title"
    ])

    if style == "upper":
        df.loc[idx, "State"] = state.upper()

    elif style == "lower":
        df.loc[idx, "State"] = state.lower()


#Different Phone Formats


for idx in df.sample(frac=0.15, random_state=4).index:

    phone = str(df.loc[idx, "Phone"])

    if phone == "None":
        continue

    option = random.randint(1,3)

    if option == 1:
        phone = "+91-" + phone

    elif option == 2:
        phone = phone[:5] + " " + phone[5:]

    else:
        phone = "+91 " + phone

    df.loc[idx, "Phone"] = phone


#Duplicate Customers (~2%)


duplicates = df.sample(n=10, random_state=5)

df = pd.concat([df, duplicates], ignore_index=True)



df.to_csv("customers_A.csv", index=False)
from IPython.display import display, FileLink


print(df.head())

print()
print("Rows :", len(df))
print("Unique Customer IDs :", df["Customer_ID"].nunique())
print("Duplicates :", len(df) - df["Customer_ID"].nunique())

   Customer_ID   Customer_Name                    Email       Phone  Gender  \
0         1001      Anvi Konda  liamchaudry@example.net  0133890838  Female   
1         1002  Nathaniel Sami    arunima23@example.com         NaN    Male   
2         1003     Ekaraj Bath   bimalabuch@example.org  5255341928  Female   
3         1004     Ekta Bhalla    azadmutti@example.com  5376724238    Male   
4         1005   Theodore Devi       wchand@example.org  9166978480    Male   

  Date_of_Birth              City        State Country   Join_Date  \
0    1984-02-04  Sultan Pur Majra       Odisha   India  2026-04-07   
1    1993-02-10         Tadipatri  West Bengal   India  2024-09-23   
2    2005-02-28            Ujjain      Manipur   India  2025-01-30   
3    1992-03-22        Bhimavaram  Maharashtra   India  2025-02-27   
4    2001-11-02            Jorhat      Tripura   India  2026-01-10   

  Loyalty_Status          Created_At        Last_Updated  
0         Silver 2026-04-28 14:50:25 2026-07-

In [11]:
import zipfile

with zipfile.ZipFile('my_data.zip', 'w') as zipf:
    zipf.write('customers_A.csv')
